# LLM2Seq Kaggle Training Notebook

Notebook này **tự chứa source code `llm2seq`**. Bạn không cần upload `llm2seq.zip` hay tạo Kaggle Dataset chứa code. Chỉ cần upload/chạy file `.ipynb` này trên Kaggle.

Workflow:

1. Giải nén source `llm2seq` đã nhúng sẵn vào `/kaggle/working/project`.
2. Cài dependencies.
3. Đăng nhập Hugging Face nếu dùng model gated/private.
4. Preprocess WikiLingua local `train.json/val.json/test.json` hoặc dataset Hugging Face sang JSONL.
5. Sinh config Kaggle đúng đường dẫn và phù hợp T4x2.
6. Train, resume, inference thử, rồi nén checkpoint/artifact.

Mặc định notebook dùng WikiLingua local JSON với schema `src: list[str]`, `tgt: list[str]`, và encoder `McGill-NLP/LLM2Vec-Sheared-LLaMA-mntp` (~1.3B), hợp lý hơn cho Kaggle T4x2 so với encoder 7B/8B.

## 1. Runtime & Cấu Hình

Trên Kaggle: bật GPU trong `Settings > Accelerator`. Nếu dùng model gated, tạo secret tên `HF_TOKEN` trong Kaggle.

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path

os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

KAGGLE_WORKING = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()

# ===== Tuỳ chỉnh nhanh =====
CONFIG_NAME = "baseline"        # baseline | mtp_only | kd_only | kd_mtp_full
RUN_PRESET = "full_under_12h"    # full_under_12h | smoke

# Input chính của bạn: WikiLingua đã tách sẵn train.json, val.json, test.json.
# Mỗi sample có dạng {"src": [câu 1, câu 2, ...], "tgt": [ý/câu tóm tắt 1, ...]}.
DATA_SOURCE = "wikilingua_json"  # wikilingua_json | hf
WIKILINGUA_JSON_DIR = None       # None = tự tìm trong /kaggle/input; hoặc set path chứa train.json/val.json/test.json

# Chỉ dùng khi DATA_SOURCE = "hf".
DATASET_NAME = "GEM/wiki_lingua"
DATASET_CONFIG = None            # chỉ dùng khi DATA_SOURCE = "hf"; local WikiLingua JSON không cần config language
SOURCE_FIELD = "source"
TARGET_FIELD = "target"

TASK_NAME = "summarization"

if RUN_PRESET == "full_under_12h":
    MAX_TRAIN = -1               # dùng toàn bộ train.json
    MAX_EVAL = 1000              # cap validation để eval nhanh, không ảnh hưởng train full data
elif RUN_PRESET == "smoke":
    MAX_TRAIN = 4000
    MAX_EVAL = 500
else:
    raise ValueError(f"Unknown RUN_PRESET: {RUN_PRESET}")

# Preset cho Kaggle T4x2: LLM2Vec chính thức, dưới 3B.
USE_T4_ENCODER = True
T4_ENCODER_NAME = "McGill-NLP/LLM2Vec-Sheared-LLaMA-mntp"  # ~1.3B, public-friendly
# Tuỳ chọn khác nếu bạn có quyền truy cập model Meta gated:
# T4_ENCODER_NAME = "McGill-NLP/LLM2Vec-Meta-Llama-32-1B-Instruct-mntp"
if RUN_PRESET == "full_under_12h":
    # Frozen 1.3B encoder + train adaptor/decoder. Designed to fit one Kaggle T4 session.
    T4_MAX_STEPS = 3000
    T4_BATCH_SIZE = 1
    T4_GRAD_ACCUM_STEPS = 16
    T4_MAX_SOURCE_LENGTH = 384
    T4_MAX_TARGET_LENGTH = 128
else:
    T4_MAX_STEPS = 300
    T4_BATCH_SIZE = 1
    T4_GRAD_ACCUM_STEPS = 8
    T4_MAX_SOURCE_LENGTH = 256
    T4_MAX_TARGET_LENGTH = 96

# Train với encoder mặc định trong config gốc, thường là 8B và nặng hơn nhiều.
FULL_BATCH_SIZE = 1
FULL_GRAD_ACCUM_STEPS = 16
FULL_MAX_STEPS = 2000
FULL_MAX_SOURCE_LENGTH = 512
FULL_MAX_TARGET_LENGTH = 256

print("Working dir:", KAGGLE_WORKING)
print("CUDA visible:", os.environ.get("CUDA_VISIBLE_DEVICES", "all"))

## 2. Dựng Source Code Từ Notebook

Cell này giải nén source `llm2seq` đã nhúng sẵn trong notebook vào `/kaggle/working/project`. Nếu muốn cập nhật code sau này, hãy tạo lại notebook từ repo local.

In [ ]:
import base64
import io
import tarfile

EMBEDDED_SOURCE_B64_PARTS = [
  "..."
]

project_root = KAGGLE_WORKING / "project"
if project_root.exists():
    shutil.rmtree(project_root)
project_root.mkdir(parents=True, exist_ok=True)

archive_bytes = base64.b64decode("".join(EMBEDDED_SOURCE_B64_PARTS))
with tarfile.open(fileobj=io.BytesIO(archive_bytes), mode="r:gz") as tar:
    tar.extractall(project_root)

os.chdir(project_root)
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

expected = project_root / "llm2seq" / "src" / "training" / "trainer.py"
if not expected.exists():
    raise FileNotFoundError(f"Không dựng được source code: {expected}")

print("PROJECT_ROOT =", project_root)
print("Embedded files =", 39)
print("Trainer =", expected)

## 3. Cài Dependencies

Kaggle thường đã có PyTorch/CUDA sẵn. Cell này không gỡ PyTorch hiện tại; nó pin `transformers==4.47.1` vì checkpoint `LLM2Vec-Sheared-LLaMA-mntp` dùng custom code còn import `LlamaFlashAttention2`, class đã bị gỡ ở các bản `transformers` mới hơn. Nếu bạn đã chạy cell này trước đó với version khác, restart kernel rồi chạy lại từ đầu.


In [ ]:
def run(cmd, cwd=None, check=True):
    print("$", cmd)
    return subprocess.run(cmd, shell=True, cwd=cwd, check=check)

run("python -m pip install -q --upgrade pip")
run('python -m pip install -q --upgrade --force-reinstall "transformers==4.47.1" "tokenizers>=0.21,<0.22"')
run("python -m pip install -q -r llm2seq/requirements.txt")

import torch
import transformers
print("torch:", torch.__version__)
print("transformers:", transformers.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu count:", torch.cuda.device_count())
    for i in range(torch.cuda.device_count()):
        print(f"gpu {i}:", torch.cuda.get_device_name(i))


## 4. Đăng Nhập Hugging Face

Không bắt buộc với `LLM2Vec-Sheared-LLaMA-mntp` nếu model public tải được, nhưng vẫn nên cấu hình secret để dùng dataset/model gated.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    secret_label = "HF_TOKEN"
    HF_TOKEN = UserSecretsClient().get_secret(secret_label)
except Exception as exc:
    HF_TOKEN = os.environ.get("HF_TOKEN")
    print("HF_TOKEN secret not available, using environment fallback:", type(exc).__name__)

if HF_TOKEN:
    from huggingface_hub import login
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("Logged in to Hugging Face.")
else:
    print("No HF token found. Public models/datasets still work; gated models may fail.")

## 5. Preprocess Dataset

Với WikiLingua local JSON, notebook đọc `train.json`, `val.json`, `test.json` có schema `src`/`tgt` là list câu, rồi ghi ra JSONL chuẩn của trainer: `source` và `target`. Nếu đổi `DATA_SOURCE = "hf"`, cell sẽ dùng script Hugging Face của project.


In [ ]:
DATA_DIR = KAGGLE_WORKING / "llm2seq_data" / "processed"
if DATA_DIR.exists():
    shutil.rmtree(DATA_DIR)
DATA_DIR.mkdir(parents=True, exist_ok=True)


def join_wikilingua_text(value):
    if value is None:
        return ""
    if isinstance(value, str):
        return value.strip()
    if isinstance(value, list):
        parts = []
        for item in value:
            text = join_wikilingua_text(item)
            if text:
                parts.append(text)
        return " ".join(parts)
    if isinstance(value, dict):
        parts = []
        for item in value.values():
            text = join_wikilingua_text(item)
            if text:
                parts.append(text)
        return " ".join(parts)
    return str(value).strip()


def normalize_wikilingua_records(raw):
    if isinstance(raw, list):
        return raw
    if isinstance(raw, dict):
        if "src" in raw and "tgt" in raw:
            return [raw]
        for key in ("data", "examples", "records"):
            if isinstance(raw.get(key), list):
                return raw[key]
        values = list(raw.values())
        if values and all(isinstance(v, dict) for v in values):
            return values
    raise ValueError("Không nhận ra cấu trúc JSON. Cần list[dict] hoặc dict có key src/tgt.")


def find_wikilingua_json_dir():
    if WIKILINGUA_JSON_DIR:
        path = Path(WIKILINGUA_JSON_DIR)
        if path.exists():
            return path
        raise FileNotFoundError(f"WIKILINGUA_JSON_DIR không tồn tại: {path}")

    candidates = [KAGGLE_WORKING]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        candidates.extend([p for p in kaggle_input.rglob("*") if p.is_dir()])

    for path in candidates:
        has_train = (path / "train.json").exists()
        has_val = (path / "val.json").exists() or (path / "validation.json").exists()
        if has_train and has_val:
            return path
    raise FileNotFoundError("Không tìm thấy thư mục chứa train.json và val.json trong /kaggle/input. Hãy Add Data dataset WikiLingua của bạn, hoặc set WIKILINGUA_JSON_DIR.")


def load_wikilingua_records(input_path):
    text = Path(input_path).read_text(encoding="utf-8-sig")
    decoder = json.JSONDecoder()
    values = []
    idx = 0
    while idx < len(text):
        while idx < len(text) and text[idx].isspace():
            idx += 1
        if idx >= len(text):
            break
        value, idx = decoder.raw_decode(text, idx)
        values.append(value)

    if len(values) == 1:
        return normalize_wikilingua_records(values[0])

    records = []
    for value in values:
        records.extend(normalize_wikilingua_records(value))
    return records


def convert_wikilingua_split(input_path, output_path, split_name, max_samples=-1):
    records = load_wikilingua_records(input_path)
    if max_samples and max_samples > 0:
        records = records[:max_samples]

    kept = 0
    with open(output_path, "w", encoding="utf-8") as f:
        for idx, example in enumerate(records):
            source = join_wikilingua_text(example.get("src") or example.get("source"))
            target = join_wikilingua_text(example.get("tgt") or example.get("target") or example.get("summary"))
            if not source or not target:
                continue
            row = {
                "id": f"{split_name}_{idx:06d}",
                "source": source,
                "target": target,
                "task": TASK_NAME,
            }
            f.write(json.dumps(row, ensure_ascii=False) + "\n")
            kept += 1
    print(f"{split_name}: {kept} examples -> {output_path}")
    return output_path


if DATA_SOURCE == "wikilingua_json":
    wiki_dir = find_wikilingua_json_dir()
    print("WikiLingua JSON dir:", wiki_dir)
    train_json = wiki_dir / "train.json"
    val_json = wiki_dir / "val.json"
    if not val_json.exists():
        val_json = wiki_dir / "validation.json"
    test_json = wiki_dir / "test.json"

    convert_wikilingua_split(train_json, DATA_DIR / "train.jsonl", "train", MAX_TRAIN)
    convert_wikilingua_split(val_json, DATA_DIR / "validation.jsonl", "validation", MAX_EVAL)
    if test_json.exists():
        convert_wikilingua_split(test_json, DATA_DIR / "test.jsonl", "test", MAX_EVAL)
else:
    preprocess_cmd = [
        "python", "-m", "llm2seq.src.data.preprocess",
        "--dataset", DATASET_NAME,
        "--output_dir", str(DATA_DIR),
        "--source_field", SOURCE_FIELD,
        "--target_field", TARGET_FIELD,
        "--task", TASK_NAME,
        "--max_train", str(MAX_TRAIN),
        "--max_eval", str(MAX_EVAL),
    ]
    if DATASET_CONFIG:
        preprocess_cmd += ["--dataset_config", DATASET_CONFIG]

    run(" ".join([repr(x) for x in preprocess_cmd]))

print("Files:", sorted(p.name for p in DATA_DIR.glob("*.jsonl")))
print("Preview:")
preview_file = DATA_DIR / "train.jsonl"
with open(preview_file, "r", encoding="utf-8") as f:
    print(f.readline()[:1200])

## 6. Sinh Config Kaggle

Cell này đọc config gốc trong `llm2seq/configs/`, vá đường dẫn dữ liệu/output, đổi encoder sang preset T4 1.3B nếu `USE_T4_ENCODER=True`, rồi lưu thành `/kaggle/working/kaggle_configs/<config>.yaml`.

In [ ]:
import yaml
from transformers import AutoConfig

base_config_path = project_root / "llm2seq" / "configs" / f"{CONFIG_NAME}.yaml"
if not base_config_path.exists():
    raise FileNotFoundError(f"Config không tồn tại: {base_config_path}")

with open(base_config_path, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

train_file = DATA_DIR / "train.jsonl"
eval_file = DATA_DIR / "validation.jsonl"
if not eval_file.exists():
    eval_file = DATA_DIR / "eval.jsonl"
if not eval_file.exists():
    eval_file = DATA_DIR / "test.jsonl"
if not train_file.exists() or not eval_file.exists():
    raise FileNotFoundError(f"Không thấy train/eval JSONL trong {DATA_DIR}: {list(DATA_DIR.glob('*.jsonl'))}")

run_name = CONFIG_NAME + ("_t4_1p3b" if USE_T4_ENCODER else "")
cfg.setdefault("project", {})["output_dir"] = str(KAGGLE_WORKING / "runs" / f"llm2seq_{run_name}")
cfg.setdefault("data", {})["train_file"] = str(train_file)
cfg.setdefault("data", {})["eval_file"] = str(eval_file)

if USE_T4_ENCODER:
    cfg["model"]["encoder_name"] = T4_ENCODER_NAME
    auto_cfg = AutoConfig.from_pretrained(T4_ENCODER_NAME, trust_remote_code=True)
    cfg["model"]["d_enc"] = int(getattr(auto_cfg, "hidden_size"))
    cfg.setdefault("adaptor", {})["use_layer_fusion"] = True
    cfg["adaptor"]["fuse_layers"] = [-1, -2, -3, -4]
    cfg.setdefault("small_decoder", {})["num_layers"] = 4
    cfg["small_decoder"]["hidden_size"] = 512
    cfg["small_decoder"]["num_heads"] = 8
    cfg["small_decoder"]["ffn_size"] = 2048
    cfg["model"]["d_dec"] = 512
    cfg.setdefault("training", {})["batch_size"] = T4_BATCH_SIZE
    cfg["training"]["grad_accum_steps"] = T4_GRAD_ACCUM_STEPS
    cfg["training"]["max_steps"] = T4_MAX_STEPS
    cfg["training"]["warmup_steps"] = min(200, max(25, T4_MAX_STEPS // 20))
    cfg["training"]["log_every_steps"] = 25
    cfg.setdefault("data", {})["max_source_length"] = T4_MAX_SOURCE_LENGTH
    cfg["data"]["max_target_length"] = T4_MAX_TARGET_LENGTH
    cfg.setdefault("evaluation", {})["eval_every_steps"] = 750 if RUN_PRESET == "full_under_12h" else 100
    cfg["evaluation"]["save_every_steps"] = 750 if RUN_PRESET == "full_under_12h" else 100
else:
    cfg.setdefault("training", {})["batch_size"] = FULL_BATCH_SIZE
    cfg["training"]["grad_accum_steps"] = FULL_GRAD_ACCUM_STEPS
    cfg["training"]["max_steps"] = FULL_MAX_STEPS
    cfg.setdefault("data", {})["max_source_length"] = FULL_MAX_SOURCE_LENGTH
    cfg["data"]["max_target_length"] = FULL_MAX_TARGET_LENGTH

# Baseline không cần resume. Các stage sau có thể resume nếu checkpoint stage trước tồn tại.
if CONFIG_NAME in {"baseline"}:
    cfg.get("training", {}).pop("resume_from", None)

CONFIG_DIR = KAGGLE_WORKING / "kaggle_configs"
CONFIG_DIR.mkdir(parents=True, exist_ok=True)
kaggle_config_path = CONFIG_DIR / f"{run_name}.yaml"
with open(kaggle_config_path, "w", encoding="utf-8") as f:
    yaml.safe_dump(cfg, f, sort_keys=False, allow_unicode=True)

print("Saved config:", kaggle_config_path)
print(yaml.safe_dump(cfg, sort_keys=False, allow_unicode=True))

## 7. Train

Chạy cell này để train theo config đã sinh. Checkpoint sẽ nằm trong `/kaggle/working/runs/...`.

Checkpoint được lưu dạng compact: không lưu frozen LLM2Vec encoder trong `best.pt/final.pt`, và chỉ giữ checkpoint định kỳ mới nhất để tránh đầy disk Kaggle.


In [ ]:
run(f"python -m llm2seq.src.training.trainer --config {str(kaggle_config_path)!r}")

## 8. Resume Training

Nếu Kaggle bị ngắt session, Add Dataset chứa checkpoint cũ hoặc copy checkpoint vào `/kaggle/working`, rồi sửa `RESUME_CHECKPOINT`.

In [ ]:
# RESUME_CHECKPOINT = "/kaggle/working/runs/llm2seq_baseline_t4_1p3b/final.pt"
# run(f"python -m llm2seq.src.training.trainer --config {str(kaggle_config_path)!r} --resume {RESUME_CHECKPOINT!r}")

## 9. Inference Nhanh Từ Checkpoint

Cell này load `best.pt` nếu có, nếu không thì load `final.pt`, rồi sinh thử output cho một input mẫu.

In [ ]:
import json
import torch
from transformers import AutoTokenizer
from llm2seq.src.models.llm2seq_model import LLM2Seq, LLM2SeqConfig
from llm2seq.src.inference.generate import autoregressive_generate

with open(kaggle_config_path, "r", encoding="utf-8") as f:
    infer_raw_cfg = yaml.safe_load(f)
infer_cfg = LLM2SeqConfig(infer_raw_cfg)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
tokenizer = AutoTokenizer.from_pretrained(infer_cfg.encoder_name, trust_remote_code=True)
if tokenizer.pad_token_id is None:
    tokenizer.pad_token = tokenizer.eos_token or tokenizer.unk_token

model = LLM2Seq(infer_cfg, vocab_size=len(tokenizer))
output_dir = Path(infer_raw_cfg["project"]["output_dir"])
ckpt_path = output_dir / "best.pt"
if not ckpt_path.exists():
    ckpt_path = output_dir / "final.pt"

if ckpt_path.exists():
    ckpt = torch.load(ckpt_path, map_location="cpu")
    model.load_state_dict(ckpt["model_state_dict"], strict=False)
    print("Loaded", ckpt_path)
else:
    print("Không tìm thấy checkpoint; model đang ở random weights.")

model.to(device)
model.eval()

# Summarization phải test bằng document dài WikiLingua, không dùng câu ngắn kiểu dịch máy.
def load_jsonl_examples(path, n=10, start=0):
    examples = []
    with open(path, "r", encoding="utf-8") as f:
        for i, line in enumerate(f):
            if i < start:
                continue
            if len(examples) >= n:
                break
            examples.append(json.loads(line))
    return examples

sample_path = DATA_DIR / "test.jsonl"
if not sample_path.exists():
    sample_path = DATA_DIR / "validation.jsonl"

NUM_SAMPLES = 10
START_INDEX = 0
examples = load_jsonl_examples(sample_path, n=NUM_SAMPLES, start=START_INDEX)
print(f"Sample file: {sample_path}")
print(f"Loaded {len(examples)} examples")

for idx, example in enumerate(examples, start=START_INDEX):
    sample_text = example["source"]
    gold_summary = example["target"]

    enc = tokenizer(
        sample_text,
        return_tensors="pt",
        truncation=True,
        max_length=infer_raw_cfg["data"]["max_source_length"],
    ).to(device)

    out_ids = autoregressive_generate(
        model,
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        max_new_tokens=128,
        temperature=0.8,
        top_k=50,
        top_p=0.9,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
        bos_token_id=tokenizer.bos_token_id or tokenizer.eos_token_id or tokenizer.pad_token_id,
    )

    pred_summary = tokenizer.decode(out_ids[0], skip_special_tokens=True)
    print("\n" + "=" * 100)
    print(f"SAMPLE {idx}")
    print("SOURCE chars:", len(sample_text))
    print("SOURCE preview:")
    print(sample_text[:1000])
    print("\nGOLD SUMMARY:")
    print(gold_summary)
    print("\nPRED SUMMARY:")
    print(pred_summary)

## 10. Đóng Gói Artifact

Kaggle giữ các file trong `/kaggle/working` sau khi notebook kết thúc. Cell này nén runs/config để tải về hoặc tạo Kaggle Dataset mới.

In [ ]:
artifact_path = KAGGLE_WORKING / f"llm2seq_{run_name}_artifacts.zip"
items = [KAGGLE_WORKING / "runs", CONFIG_DIR]
existing = [str(p) for p in items if p.exists()]
if artifact_path.exists():
    artifact_path.unlink()
run(f"zip -r -q {str(artifact_path)!r} " + " ".join(repr(p) for p in existing))
print("Artifact:", artifact_path)